# Manual Testing: Financial Chat Agent

This notebook allows you to manually test the chat agent API by sending multiple questions for one user and viewing the answers.

In [1]:
# Define questions for API testing
questions = [
    "I'm a medium risk investor with a 2 year horizon. Give me a summary and overview for Atul Auto. Include key financial ratios.",
    "Compare Atul Auto with Ashok Leyland. Provide valuation, pros/cons, and latest news items.",
    "My budget is 1 lakh. Should I buy now or wait? Keep it concise and which one from the above two stocks is a better choice for me.",
    "what was my previous message.",
]

In [2]:
import requests

api_url = "http://localhost:8000/chat"
user_id = "manualtestuser"
chat_id = "manualchat1"

api_answers = []
for i, question in enumerate(questions):
    print(f"Sending Question {i+1}: {question}")
    payload = {"user_id": user_id, "chat_id": chat_id, "message": question}
    response = requests.post(api_url, json=payload)
    if response.status_code == 200:
        answer = response.json().get("answer", "No answer returned")
        print(f"API Answer: {answer}\n")
        api_answers.append(answer)
    else:
        print(f"API Error: {response.status_code} - {response.text}\n")
        api_answers.append(None)

Sending Question 1: I'm a medium risk investor with a 2 year horizon. Give me a summary and overview for Atul Auto. Include key financial ratios.
API Answer: ### Atul Auto Overview

Atul Auto is engaged in manufacturing three-wheeled vehicles and has demonstrated steady performance in the Indian auto sector. Here’s a summary of key financial metrics that characterize its current position:

| Metric                          | Value           | Unit        |
|---------------------------------|----------------|-------------|
| Market Capitalization            | 1,179.85       | INR Cr      |
| PE Ratio                        | 50.73          | -           |
| PBV Ratio                       | 2.68           | -           |
| Dividend Yield                  | 0.00           | %           |
| EPS                             | 8.38           | INR         |
| ROA (Return on Assets)         | 2.89           | %           |
| ROE (Return on Equity)         | 4.91           | %           |
| RO

In [13]:
from python_helpers import get_co_code

api_key = "YOUR_BEARER_TOKEN_HERE"  # Replace with your actual token
company_name = "Reliance"
co_code = get_co_code(company_name, api_key)
print(f"co_code for '{company_name}':", co_code)

Error fetching co_code: 500 Server Error: Internal Server Error for url: https://insbaapis.cmots.com/api/CompanyMaster
co_code for 'Reliance': None


In [14]:
# Debug: Check what's in Mongo messages and response_items
import os, json
from pymongo import MongoClient
from pprint import pprint

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("DB_NAME", "chatbotdb")
USER_ID = "manualtestuser"
CHAT_ID = "manualchat1"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

# Check simple messages collection
messages_col = db["messages"]
msg_docs = list(messages_col.find({"user_id": USER_ID, "chat_id": CHAT_ID}).sort("ts", 1))
print(f"Found {len(msg_docs)} messages for user_id={USER_ID} chat_id={CHAT_ID}")
for i, doc in enumerate(msg_docs[-5:]):
    print(f"  {i+1}. {doc.get('role')} [{doc.get('ts')}]: {doc.get('text', '')[:100]}...")

# Check response_items collection
items_col = db["response_items"]
item_docs = list(items_col.find({"user_id": USER_ID, "chat_id": CHAT_ID}).sort("ts", 1))
print(f"\nFound {len(item_docs)} response items for user_id={USER_ID} chat_id={CHAT_ID}")
for i, d in enumerate(item_docs[-5:]):
    it = d.get("item", {})
    t = it.get("type")
    c = it.get("content")
    c_preview = (c[:100] + "...") if isinstance(c, str) and len(c) > 100 else c
    print(f"  {i+1}. type={t} | content={c_preview}")

print(f"\nAll collections in {DB_NAME}: {db.list_collection_names()}")

Found 60 messages for user_id=manualtestuser chat_id=manualchat1
  1. assistant [1755298336163]: Your last question was not specified in the current context. If you have a specific inquiry or topic...
  2. user [1755298384774]: Between the above two stocks which is better for a 1 lakh budget?...
  3. assistant [1755298384776]: Could you please specify the two stocks you're referring to?...
  4. user [1755298423054]: Between the above two stocks which is better for a 1 lakh budget?...
  5. assistant [1755298423055]: Could you please specify which two stocks you are referring to?...

Found 0 response items for user_id=manualtestuser chat_id=manualchat1

All collections in chatbotdb: ['messages', 'msg_vectors', 'response_items']


In [9]:
# Test with specific reference to the two stocks that were discussed
import requests

test_payload = {
    "user_id": "manualtestuser", 
    "chat_id": "manualchat1", 
    "message": "Between the above two stocks which is better for a 1 lakh budget?"
}

response = requests.post("http://localhost:8000/chat", json=test_payload)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    print(f"Answer: {response.json().get('answer')}")
else:
    print(f"Error: {response.text}")

Status: 200
Answer: Could you please specify which two stocks you are referring to?


In [12]:
# Test the context hint fix
import requests

response = requests.post("http://localhost:8000/chat", json={
    "user_id": "freshtest",
    "chat_id": "freshchat", 
    "message": "Which of the above two stocks should I choose for 1 lakh investment?"
})

if response.status_code == 200:
    answer = response.json().get("answer", "")
    print(f"Answer: {answer}")
else:
    print(f"Error: {response.status_code} - {response.text}")

Answer: Here's a comparison of Atul Auto and Ashok Leyland based on their latest TTM (Trailing Twelve Months) financial metrics:

### Atul Auto vs. Ashok Leyland

| Metric                 | Atul Auto                  | Ashok Leyland              |
|------------------------|---------------------------|----------------------------|
| **Market Capitalization** | ₹1,179.85 Cr             | ₹71,621.9 Cr               |
| **PE Ratio**           | 50.73                     | 22.42                      |
| **EPS**                | ₹8.38                     | ₹5.44                      |
| **ROE**                | 4.91%                     | 25.40%                     |
| **ROA**                | 2.89%                     | 4.27%                      |
| **Current Ratio**      | 2.69                      | 2.60                       |
| **Debt to Equity Ratio** | 0.25                     | 4.08                       |
| **Net Profit Margin**  | 3.19%                     | 6.50%                 

In [ ]:
# Test generic company name extraction with different stocks
import requests
from pymongo import MongoClient
import os

# Clear and test with completely different companies
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("DB_NAME", "chatbotdb")
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
db["messages"].delete_many({"user_id": "generictest", "chat_id": "genericchat"})

# Test sequence with different companies
test_sequence = [
    "Compare Reliance Industries with Tata Motors for investment",
    "Which of the above two companies has better growth potential?"
]

for i, q in enumerate(test_sequence):
    print(f"\nQ{i+1}: {q}")
    response = requests.post("http://localhost:8000/chat", json={
        "user_id": "generictest",
        "chat_id": "genericchat", 
        "message": q
    })
    if response.status_code == 200:
        answer = response.json().get("answer", "")
        print(f"A{i+1}: {answer[:300]}...")
    else:
        print(f"Error: {response.status_code}")